In [1]:
import pandas as pd
import numpy as np
import os

Loading the DataSets

In [2]:
INPUT_DIR  = '/Users/abhishekkarthikakunuru/Desktop/Data Engineering /Datasets/Original Datasets'
OUTPUT_DIR = '/Users/abhishekkarthikakunuru/Desktop/Data Engineering /Datasets/Cleaned Datasets'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Berlin/Brandenburg bounding box
LAT_MIN, LAT_MAX = 51.2, 53.6
LON_MIN, LON_MAX = 11.2, 14.8

In [4]:
# GTFS route_type → human readable label
ROUTE_TYPE_MAP = {
    0:    "Tram",
    1:    "Metro / U-Bahn",
    2:    "Rail",
    3:    "Bus",
    4:    "Ferry",
    5:    "Cable Car",
    7:    "Funicular",
    100:  "Rail (Regional)",
    106:  "Rail (Regional)",
    109:  "S-Bahn",
    400:  "U-Bahn",
    700:  "Bus",
    900:  "Tram",
    1000: "Water Transport",
}

PATHWAY_MODE_MAP = {
    1: "Walkway",
    2: "Stairs",
    3: "Moving Sidewalk",
    4: "Escalator",
    5: "Elevator",
    6: "Fare Gate",
    7: "Exit Gate",
}

LOCATION_TYPE_MAP = {
    0: "Stop / Platform",
    1: "Station",
    2: "Entrance / Exit",
    3: "Generic Node",
    4: "Boarding Area",
}

In [5]:
def load(name):
    path = os.path.join(INPUT_DIR, f"{name}.csv")
    df   = pd.read_csv(path, low_memory=False)
    df.columns = (df.columns
                    .str.strip()
                    .str.replace('"', '', regex=False)
                    .str.lower())
    return df
def save(df, name, notes):
    path = os.path.join(OUTPUT_DIR, f"{name}_clean.csv")
    df.to_csv(path, index=False)
    print(f" Saved {name}_clean.csv  ({len(df):,} rows)  — {notes}")


def report(name, before, after, issues):
    removed = before - after
    print(f"\n{'='*55}")
    print(f"  {name.upper()}")
    print(f"{'='*55}")
    print(f"  Rows before : {before:,}")
    print(f"  Rows after  : {after:,}  (removed {removed:,})")
    for issue in issues:
        print(f"  ✔  {issue}")

1.Agency Dataset

In [6]:
def clean_agency():
    df = load("agency")
    before = len(df)
    issues = []

    missing_lang = df["agency_lang"].isnull().sum()
    df["agency_lang"] = df["agency_lang"].fillna("de")
    issues.append(f"Filled {missing_lang} missing agency_lang with 'de'")

    missing_phone = df["agency_phone"].isnull().sum()
    df["agency_phone"] = df["agency_phone"].fillna("N/A")
    issues.append(f"Filled {missing_phone} missing agency_phone with 'N/A'")

    df["agency_name"] = df["agency_name"].str.strip()
    issues.append("Stripped whitespace from agency_name")

    report("agency", before, len(df), issues)
    save(df, "agency", "lang & phone nulls filled")
    return df

2.Routes

In [7]:
def clean_routes():
    df = load("routes")
    before = len(df)
    issues = []

    missing_name = df["route_long_name"].isnull().sum()
    df["route_long_name"] = df["route_long_name"].fillna(df["route_short_name"])
    issues.append(f"Filled {missing_name} missing route_long_name with route_short_name")

    missing_desc = df["route_desc"].isnull().sum()
    df["route_desc"] = df["route_desc"].fillna("No description")
    issues.append(f"Filled {missing_desc} missing route_desc")

    missing_color = df["route_color"].isnull().sum()
    df["route_color"]      = df["route_color"].fillna("FFFFFF")
    df["route_text_color"] = df["route_text_color"].fillna("000000")
    issues.append(f"Filled {missing_color} missing route_color/route_text_color with defaults")

    df["transport_mode"] = df["route_type"].map(ROUTE_TYPE_MAP).fillna("Unknown")
    issues.append("Added transport_mode column from route_type")

    df["route_short_name"] = df["route_short_name"].astype(str).str.strip()
    issues.append("Cleaned route_short_name to string")

    report("routes", before, len(df), issues)
    save(df, "routes", "names, colors, transport_mode added")
    return df

3.Stops

In [8]:
def clean_stops():
    df = load("stops")
    before = len(df)
    issues = []

    df["stop_lat"] = pd.to_numeric(df["stop_lat"], errors="coerce")
    df["stop_lon"] = pd.to_numeric(df["stop_lon"], errors="coerce")

    out_of_bbox = (~(df["stop_lat"].between(LAT_MIN, LAT_MAX) &
                     df["stop_lon"].between(LON_MIN, LON_MAX))).sum()
    df = df[
        df["stop_lat"].between(LAT_MIN, LAT_MAX) &
        df["stop_lon"].between(LON_MIN, LON_MAX)
    ]
    issues.append(f"Removed {out_of_bbox} stops outside Berlin/Brandenburg bbox")

    df["stop_code"]     = df["stop_code"].fillna("N/A")
    df["stop_desc"]     = df["stop_desc"].fillna("No description")
    df["platform_code"] = df["platform_code"].fillna("N/A")
    df["zone_id"]       = df["zone_id"].fillna("Unknown")
    issues.append("Filled stop_code, stop_desc, platform_code, zone_id nulls")

    issues.append("parent_station nulls kept (null = top-level station per GTFS spec)")

    df["location_type"]      = pd.to_numeric(df["location_type"], errors="coerce").fillna(0).astype(int)
    df["location_type_name"] = df["location_type"].map(LOCATION_TYPE_MAP).fillna("Stop / Platform")
    issues.append("Added location_type_name column")

    df["wheelchair_boarding"] = pd.to_numeric(df["wheelchair_boarding"], errors="coerce").fillna(0).astype(int)
    df["is_accessible"] = df["wheelchair_boarding"] == 1
    issues.append("Added is_accessible boolean column")

    df["stop_lat"] = df["stop_lat"].round(6)
    df["stop_lon"] = df["stop_lon"].round(6)
    issues.append("Rounded coordinates to 6 decimal places")

    report("stops", before, len(df), issues)
    save(df, "stops", "coords validated, types labeled, accessibility flagged")
    return df

4. Calender

In [9]:
def clean_calendar():
    df = load("calendar")
    before = len(df)
    issues = []

    df["start_date"] = pd.to_datetime(df["start_date"].astype(str), format="%Y%m%d")
    df["end_date"]   = pd.to_datetime(df["end_date"].astype(str),   format="%Y%m%d")
    issues.append("Converted start_date & end_date from integer to datetime")

    df["service_duration_days"] = (df["end_date"] - df["start_date"]).dt.days
    issues.append("Added service_duration_days column")

    df["weekday_service_days"]        = df[["monday","tuesday","wednesday","thursday","friday"]].sum(axis=1)
    df["weekend_service_days"]        = df[["saturday","sunday"]].sum(axis=1)
    df["total_service_days_per_week"] = df[["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]].sum(axis=1)

    def tag_service(row):
        if row["weekday_service_days"] > 0 and row["weekend_service_days"] > 0:
            return "Full Week"
        elif row["weekday_service_days"] > 0:
            return "Weekdays Only"
        elif row["weekend_service_days"] > 0:
            return "Weekends Only"
        else:
            return "No Service"

    df["service_pattern"] = df.apply(tag_service, axis=1)
    issues.append("Added weekday_service_days, weekend_service_days, service_pattern columns")

    report("calendar", before, len(df), issues)
    save(df, "calendar", "dates converted, service patterns labeled")
    return df


5. Calender Dates

In [10]:
def clean_calendar_dates():
    df = load("calendar_dates")
    before = len(df)
    issues = []

    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
    issues.append("Converted date from integer to datetime")

    df["exception_label"] = df["exception_type"].map({1: "Service Added", 2: "Service Removed"})
    issues.append("Added exception_label column")

    df["day_of_week"] = df["date"].dt.day_name()
    df["month"]       = df["date"].dt.month_name()
    df["year"]        = df["date"].dt.year
    issues.append("Added day_of_week, month, year columns")

    report("calendar_dates", before, len(df), issues)
    save(df, "calendar_dates", "dates converted, exception labels added")
    return df


6. Transfers

In [11]:
def clean_transfers():
    df = load("transfers")
    before = len(df)
    issues = []

    median_time  = df["min_transfer_time"].median()
    missing_time = df["min_transfer_time"].isnull().sum()
    df["min_transfer_time"] = df["min_transfer_time"].fillna(median_time).astype(int)
    issues.append(f"Filled {missing_time} missing min_transfer_time with median ({int(median_time)}s)")

    df["min_transfer_minutes"] = (df["min_transfer_time"] / 60).round(1)
    issues.append("Added min_transfer_minutes column")

    transfer_type_map = {
        0: "Recommended",
        1: "Timed Transfer",
        2: "Minimum Time Required",
        3: "No Transfer Possible",
    }
    df["transfer_type_label"] = df["transfer_type"].map(transfer_type_map).fillna("Unknown")
    issues.append("Added transfer_type_label column")

    for col in ["from_route_id", "to_route_id", "from_trip_id", "to_trip_id"]:
        df[col] = df[col].fillna("N/A")
    issues.append("Filled optional route_id/trip_id nulls with N/A")

    df["is_same_stop"] = df["from_stop_id"] == df["to_stop_id"]
    issues.append("Added is_same_stop flag")

    report("transfers", before, len(df), issues)
    save(df, "transfers", "transfer times filled, labels added")
    return df

7. Pathways

In [12]:
def clean_pathways():
    df = load("pathways")
    before = len(df)
    issues = []

    median_tt  = df["traversal_time"].median()
    missing_tt = df["traversal_time"].isnull().sum()
    df["traversal_time"] = df["traversal_time"].fillna(median_tt).astype(int)
    issues.append(f"Filled {missing_tt} missing traversal_time with median ({int(median_tt)}s)")

    missing_len = df["length"].isnull().sum()
    df["length"] = pd.to_numeric(df["length"], errors="coerce").fillna(0).round(1)
    issues.append(f"Filled {missing_len} missing length with 0")

    all_null_cols = [c for c in df.columns if df[c].isnull().all()]
    if all_null_cols:
        df = df.drop(columns=all_null_cols)
        issues.append(f"Dropped {len(all_null_cols)} fully null columns: {all_null_cols}")

    df["pathway_mode_name"] = df["pathway_mode"].map(PATHWAY_MODE_MAP).fillna("Unknown")
    issues.append("Added pathway_mode_name column")

    df["direction"] = df["is_bidirectional"].map({0: "One-Way", 1: "Two-Way"})
    issues.append("Added direction column (One-Way / Two-Way)")

    report("pathways", before, len(df), issues)
    save(df, "pathways", "nulls filled, mode/direction labels added, dead columns dropped")
    return df

8. Levels

In [13]:
def clean_levels():
    df = load("levels")
    before = len(df)
    issues = []

    missing_names = df["level_name"].isnull().sum()
    df["level_name"] = df["level_name"].fillna("Unnamed Level")
    issues.append(f"Filled {missing_names} missing level_name with 'Unnamed Level'")

    def level_position(idx):
        if idx > 0:   return "Above Ground"
        elif idx < 0: return "Below Ground"
        else:         return "Ground Level"

    df["level_position"] = df["level_index"].apply(level_position)
    issues.append("Added level_position column (Above / Below / Ground)")

    report("levels", before, len(df), issues)
    save(df, "levels", "names filled, position labeled")
    return df


SUMMARY REPORT

In [14]:
def print_summary(results):
    print("\n" + "="*55)
    print("  CLEANING COMPLETE — SUMMARY")
    print("="*55)
    for name, df in results.items():
        print(f"  {name:<20} → {len(df):>7,} rows  ✅")
    total = sum(len(df) for df in results.values())
    print(f"\n  Total clean rows : {total:,}")
    print(f"  Output folder    : ~/Desktop/ML/vbb_clean/")
    print("\n  Files ready for:")
    print("  → PostgreSQL ingestion")
    print("  → Apache Spark processing")
    print("  → Kafka streaming")
    print("  → Grafana / Plotly visualization")
    print("="*55)

MAIN

In [15]:
if __name__ == "__main__":
    print("\n🚇 VBB DATA CLEANING PIPELINE")
    print("   Input  : ~/Desktop/ML/")
    print("   Output : ~/Desktop/ML/vbb_clean/")
    print("   Cleaning all 8 GTFS datasets...\n")

    results = {
        "agency":          clean_agency(),
        "routes":          clean_routes(),
        "stops":           clean_stops(),
        "calendar":        clean_calendar(),
        "calendar_dates":  clean_calendar_dates(),
        "transfers":       clean_transfers(),
        "pathways":        clean_pathways(),
        "levels":          clean_levels(),
    }

    print_summary(results)



🚇 VBB DATA CLEANING PIPELINE
   Input  : ~/Desktop/ML/
   Output : ~/Desktop/ML/vbb_clean/
   Cleaning all 8 GTFS datasets...


  AGENCY
  Rows before : 35
  Rows after  : 35  (removed 0)
  ✔  Filled 2 missing agency_lang with 'de'
  ✔  Filled 33 missing agency_phone with 'N/A'
  ✔  Stripped whitespace from agency_name
 Saved agency_clean.csv  (35 rows)  — lang & phone nulls filled

  ROUTES
  Rows before : 1,322
  Rows after  : 1,322  (removed 0)
  ✔  Filled 1321 missing route_long_name with route_short_name
  ✔  Filled 1321 missing route_desc
  ✔  Filled 1203 missing route_color/route_text_color with defaults
  ✔  Added transport_mode column from route_type
  ✔  Cleaned route_short_name to string
 Saved routes_clean.csv  (1,322 rows)  — names, colors, transport_mode added

  STOPS
  Rows before : 41,986
  Rows after  : 41,840  (removed 146)
  ✔  Removed 146 stops outside Berlin/Brandenburg bbox
  ✔  Filled stop_code, stop_desc, platform_code, zone_id nulls
  ✔  parent_station nulls 